In [ ]:
from pathlib import Path
from tqdm import tqdm

import pandas as pd
import matplotlib.pyplot as plt 
import numpy as np
from plotly import express as px

import timm
import torch
from torch import nn
from torch.nn import functional as F
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import Dataset,DataLoader, random_split
from torchvision.transforms.v2 import ToImage, Compose, ToDtype, Resize
from torchvision.utils import save_image
from timm.models import create_model
from torchkeras import KerasModel

from scipy.ndimage import center_of_mass, shift

# 载入数据集

In [ ]:
data_dir = "../data/wf-less"
# file_list = [11,12,13,14]
file_list = [11]
all_data = []
for pkl_file in tqdm(Path(data_dir).glob('*.pkl')):
    data = pd.read_pickle(pkl_file, compression="zip")
    data["patch"] = pkl_file
    all_data.append(data)
data = pd.concat(all_data, axis=0)
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 82014 entries, 0 to 6000
Data columns (total 10 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   J       82014 non-null  float64
 1   pib     82014 non-null  float64
 2   _v      82014 non-null  object 
 3   _img    82014 non-null  object 
 4   _diff   82014 non-null  float64
 5   gamma   82014 non-null  float64
 6   r       82014 non-null  float64
 7   delta   82014 non-null  float64
 8   _epoch  82014 non-null  int64  
 9   patch   82014 non-null  object 
dtypes: float64(6), int64(1), object(3)
memory usage: 6.9+ MB


## 数据分析

In [6]:
px.imshow(data.iloc[0]._img)

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

Figure({
    'data': [{'coloraxis': 'coloraxis',
              'hovertemplate': 'x: %{x}<br>y: %{y}<br>color: %{z}<extra></extra>',
              'name': '0',
              'type': 'heatmap',
              'xaxis': 'x',
              'yaxis': 'y',
              'z': {'bdata': ('AgICAQECAQAAAgEBAAABAgEBAgEBAQ' ... 'IAAQIAAQECAAEBAQAAAAAAAQICAA=='),
                    'dtype': 'u1',
                    'shape': '272, 272'}}],
    'layout': {'coloraxis': {'colorscale': [[0.0, '#0d0887'], [0.1111111111111111,
                                            '#46039f'], [0.2222222222222222,
                                            '#7201a8'], [0.3333333333333333,
                                            '#9c179e'], [0.4444444444444444,
                                            '#bd3786'], [0.5555555555555556,
                                            '#d8576b'], [0.6666666666666666,
                                            '#ed7953'], [0.7777777777777778,
                                            '#fb9f3a'], [0.8888888888888888,
                                            '#fdca26'], [1.0, '#f0f921']]},
               'margin': {'t': 60},
               'template': '...',
               'xaxis': {'anchor': 'y', 'constrain': 'domain', 'domain': [0.0, 1.0], 'scaleanchor': 'y'},
               'yaxis': {'anchor': 'x', 'autorange': 'reversed', 'constrain': 'domain', 'domain': [0.0, 1.0]}}
})

In [7]:
best_iter = data.J.argmax()
best_iter = data.iloc[best_iter]

px.imshow(best_iter._img)

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

Figure({
    'data': [{'coloraxis': 'coloraxis',
              'hovertemplate': 'x: %{x}<br>y: %{y}<br>color: %{z}<extra></extra>',
              'name': '0',
              'type': 'heatmap',
              'xaxis': 'x',
              'yaxis': 'y',
              'z': {'bdata': ('AgICAgEBAgIBAQEBAQEBAQIBAgEBAQ' ... 'EBAQAAAgICAQEBAAEBAQABAQIBAA=='),
                    'dtype': 'u1',
                    'shape': '272, 272'}}],
    'layout': {'coloraxis': {'colorscale': [[0.0, '#0d0887'], [0.1111111111111111,
                                            '#46039f'], [0.2222222222222222,
                                            '#7201a8'], [0.3333333333333333,
                                            '#9c179e'], [0.4444444444444444,
                                            '#bd3786'], [0.5555555555555556,
                                            '#d8576b'], [0.6666666666666666,
                                            '#ed7953'], [0.7777777777777778,
                                            '#fb9f3a'], [0.8888888888888888,
                                            '#fdca26'], [1.0, '#f0f921']]},
               'margin': {'t': 60},
               'template': '...',
               'xaxis': {'anchor': 'y', 'constrain': 'domain', 'domain': [0.0, 1.0], 'scaleanchor': 'y'},
               'yaxis': {'anchor': 'x', 'autorange': 'reversed', 'constrain': 'domain', 'domain': [0.0, 1.0]}}
})

# 训练模型

In [32]:
def move_mc_to_center(img):
    cy, cx = img.shape
    cy, cx = cy/2, cx/2
    my, mx = center_of_mass(img)
    dy, dx = int(cy-my), int(cx-mx)
    
    return shift(img, (dy, dx))

def resize(img, size):
    h, w = img.shape
    if w >= size and h >= size:
        return img[int(h/2-size/2):int(h/2+size/2), int(w/2-size/2):int(w/2+size/2)]
    else:
        px = max(0, int(size/2-w/2))
        py = max(0, int(size/2-h/2))
        return np.pad(img, ((py, py),(px, px)), mode='edge')

In [33]:
resize(move_mc_to_center(np.random.rand(250,240)), 300).shape

(300, 300)

In [34]:
best_iter = data.J.argmax()
best_iter = data.iloc[best_iter]

px.imshow(resize(move_mc_to_center(best_iter._img), 300))

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

Figure({
    'data': [{'coloraxis': 'coloraxis',
              'hovertemplate': 'x: %{x}<br>y: %{y}<br>color: %{z}<extra></extra>',
              'name': '0',
              'type': 'heatmap',
              'xaxis': 'x',
              'yaxis': 'y',
              'z': {'bdata': ('AAAAAAAAAAAAAAAAAAAAAAAAAAAAAA' ... 'EBAQIBAQEBAQEBAQEBAQEBAQEBAQEB'),
                    'dtype': 'u1',
                    'shape': '300, 300'}}],
    'layout': {'coloraxis': {'colorscale': [[0.0, '#0d0887'], [0.1111111111111111,
                                            '#46039f'], [0.2222222222222222,
                                            '#7201a8'], [0.3333333333333333,
                                            '#9c179e'], [0.4444444444444444,
                                            '#bd3786'], [0.5555555555555556,
                                            '#d8576b'], [0.6666666666666666,
                                            '#ed7953'], [0.7777777777777778,
                                            '#fb9f3a'], [0.8888888888888888,
                                            '#fdca26'], [1.0, '#f0f921']]},
               'margin': {'t': 60},
               'template': '...',
               'xaxis': {'anchor': 'y', 'constrain': 'domain', 'domain': [0.0, 1.0], 'scaleanchor': 'y'},
               'yaxis': {'anchor': 'x', 'autorange': 'reversed', 'constrain': 'domain', 'domain': [0.0, 1.0]}}
})

# 训练模型

In [35]:
BATCH_SIZE = 128
IMG_SIZE = 300

class AOShapingDataset(Dataset):
    def __init__(self, data, img_size=IMG_SIZE, transform=None):
        self.data = data
        self.transform = transform
        self.img_size = img_size
        
        # best_iter = data.iloc[data.pib.argmax()]
        # self.target_v = best_iter._v
        
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        sample = self.data.iloc[idx]
        img = resize(move_mc_to_center(sample._img), self.img_size)
        assert img.shape == (self.img_size, self.img_size), f"{sample._img.shape}"
        v = sample._v/ 500
        if self.transform:
            img = self.transform(img)
        return img, torch.tensor(v, dtype=torch.float32)
    
transforms = Compose([ToImage(), ToDtype(torch.float32, scale=True)])

dataset = AOShapingDataset(data, transform=transforms)
train_dataset, test_dataset = random_split(dataset, [0.8,0.2])
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

## 选取主干网

In [ ]:
def train(model_name):
    assert model_name in timm.list_models()

    torch.cuda.empty_cache()
    net = create_model(model_name, in_chans=1, num_classes=64, pretrained=False)
    ckpt_path = f'../ckpts/{model_name}'
    Path(ckpt_path).mkdir(parents=True, exist_ok=True)
    optimizer = Adam(net.parameters(), lr=1e-3)
    model = KerasModel(net, optimizer=optimizer, loss_fn=torch.nn.MSELoss(), lr_scheduler=CosineAnnealingLR(optimizer, T_max=10, eta_min=1e-6))
    model.fit(train_loader, test_loader, epochs=20, patience=5, ckpt_path=ckpt_path+'/checkpoints')
    return model

In [ ]:
model = train('mobilenetv5_base')

In [ ]:
%%time
model = train('vgg13')

In [ ]:
%%time
model = train('mambaout_kobe')

### visualize with grad cam

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image




## DM仿真代理
 1. vae
 2. vqgan

In [36]:
class CVAE(nn.Module):
    def __init__(self, img_size=IMG_SIZE, condition_dim=64):
        super(CVAE, self).__init__()
        input_dim = img_size**2 + condition_dim
        self.img_dim = img_size**2
        self.condition_dim = condition_dim
        
        self.fc1 = nn.Linear(input_dim, 400)
        self.fc21 = nn.Linear(400, 20)
        self.fc22 = nn.Linear(400, 20)
        self.fc3 = nn.Linear(20+condition_dim, 400)
        self.fc4 = nn.Linear(400, input_dim)
        
    def encode(self, x, y):
        #输入样本和标签y的one-hot向量连接
        con = torch.cat((x, y), 1)
        h1 = F.relu(self.fc1(con))
        return self.fc21(h1), self.fc22(h1)

    def reparameterize(self, mu, logvar):
        #训练时使用重参数化技巧，测试时不用。（测试时应该可以用）
        if self.training:
            std = torch.exp(0.5*logvar)
            eps = torch.randn_like(std)
            return eps.mul(std).add_(mu)
        else:
            return mu

    def decode(self, z, y):
        #解码器的输入：将z和y的one-hot向量连接
        cat = torch.cat((z, y), 1)
        h3 = F.relu(self.fc3(cat))
        return F.sigmoid(self.fc4(h3))

    def forward(self, x, y):
        batch_size = x.shape[0]
        mu, logvar = self.encode(x.view(batch_size, -1), y)
        z = self.reparameterize(mu, logvar)
        return self.decode(z, y), mu, logvar

# Reconstruction + KL divergence losses summed over all elements and batch
def loss_function(recon_x, x, mu, logvar):
    BCE = F.mse_loss(recon_x, x, reduction='sum')

    # see Appendix B from VAE paper:
    # Kingma and Welling. Auto-Encoding Variational Bayes. ICLR, 2014
    # https://arxiv.org/abs/1312.6114
    # 0.5 * sum(1 + log(sigma^2) - mu^2 - sigma^2)
    KLD = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())

    return BCE + KLD

model = CVAE().to('cuda')
optimizer = Adam(model.parameters(), lr=1e-3)

In [ ]:
def train(epoch, device='cuda'):
    #Sets the module in training mode.
    model.train()
    train_loss = 0

    for batch_idx, (img, voltages) in enumerate(train_loader): 
        img = img.to(device)
        voltages = voltages.to(device)
        optimizer.zero_grad()
        
        recon_batch, mu, logvar = model(img, voltages)
        #训练样本展平，在每个样本后面连接标签的one-hot向量
        flat_data = img.view(-1, img.shape[2]*img.shape[3])
        con = torch.cat((flat_data, voltages), 1)
        
        loss = loss_function(recon_batch, con, mu, logvar)
        loss.backward()
        train_loss += loss.item()
        optimizer.step()

        if batch_idx % 10 == 0:
            print('Train Epoch: {} [{}/{} ({:.0f}%)]\tLoss: {:.6f}'.format(
                epoch,
                batch_idx * len(img),
                len(train_loader.dataset),
                100. * batch_idx / len(train_loader),
                loss.item() / len(img)), end="\r")
    print('====> Epoch: {} Average loss: {:.4f}'.format(
          epoch, train_loss / len(train_loader.dataset)))
    
def test(epoch, device='cuda:0'):
    #Sets the module in evaluation mode
    model.eval()
    test_loss = 0

    with torch.no_grad():
        for i, (img, voltages) in enumerate(test_loader):
            img, voltages = img.to(device), voltages.to(device)
            recon_batch, mu, logvar = model(img, voltages)
            
            flat_data = img.view(-1, img.shape[2]*img.shape[3])
            con = torch.cat((flat_data, voltages), 1)
            test_loss += loss_function(recon_batch, con, mu, logvar).item()

            if i == 0:
                n = min(img.size(0), 8)
                recon_image = recon_batch[:, 0:recon_batch.shape[1]-voltages.shape[-1]]
                print(recon_image.shape)
                recon_image = recon_image.view(BATCH_SIZE, 1, *img.shape[-2:])
                print('---',recon_image.shape)
                comparison = torch.cat([img[:n],
                                      recon_image.view(BATCH_SIZE, 1, *img.shape[-2:])[:n]])
                save_image(comparison.cpu(),
                         '../logs/vae/' + str(epoch) + '.png', nrow=n)

    test_loss /= len(test_loader.dataset)
    print('====> Test set loss: {:.4f}'.format(test_loss))
    return test_loss

EPOCH = 20
for epoch in range(1, EPOCH + 1):
    train(epoch)
    test_loss = test(epoch)
    if epoch % 10 == 0 or epoch == EPOCH:
        path = f'../ckpts/vae/{epoch}-{test_loss:.4e}.pt'
        print(f'save model @ {path}')
        torch.save(model.state_dict(), path)

In [ ]:
# from PIL import Image

with torch.no_grad():
    #采样过程
    z = torch.randn(1, 20).to('cuda')
   
    c = np.zeros(shape=(z.shape[0], 64))
    c = torch.FloatTensor(c).to('cuda')
    sample = model.decode(z, c).cpu()
    #模型的输出矩阵：每一行的末尾都加了one-hot向量，要去掉这个one-hot向量再转换为图片。
    generated_image = sample[:, 0:sample.shape[1]-64]
    
    # img_grid = Image.fromarray(make_grid(generated_image.view(8, 1, 300, 300)).cpu().numpy())
    # img_grid.show()
    
    img = generated_image.view(-1, 300, 300).cpu().numpy()[0]
    
    plt.imshow(img)
    